# 🌊 FloodSight-UQ West Coast India — RF + 4-Method Uncertainty Quantification
## Flood Susceptibility Mapping · GEE Python API · Google Colab

**Study area:** All coastal districts along India's West Coast (Arabian Sea coastline)
**States:** Gujarat · Maharashtra · Goa · Karnataka · Kerala · Tamil Nadu

| Design choice | Rationale |
|---|---|
| **Single model: Random Forest** | Sufficient for large-scale studies; ensemble of internal trees provides built-in variance |
| **17 conditioning factors (no SAR predictors)** | SAR is already used to construct flood inventory — using it again as a predictor introduces circular data leakage |
| **4 UQ methods replacing Shannon entropy / inter-model StdDev** | Hydrology-standard uncertainty characterisation with formal coverage guarantees via conformal prediction |

### 🧮 Uncertainty Quantification Framework
| # | Method | Hydrology context | Output range |
|---|--------|------------------|-------------|
| **UQ-1** | **Bernoulli Variance** `p(1−p)` | Aleatory uncertainty of binary flood prediction; standard in ML-based FSM papers | [0, 0.25] |
| **UQ-2** | **Margin Confidence** `min(p, 1−p)` | Distance from the 0.5 decision boundary; proxy for model confidence | [0, 0.50] |
| **UQ-3** | **Bootstrap RF Variance** | Epistemic / parameter uncertainty via K bootstrap sub-samples of training data | [0, …] |
| **UQ-4** | **Conformal Prediction Sets** | Distribution-free coverage guarantee at user-defined significance α; yields spatial *prediction sets* that are provably valid under exchangeability | Categorical |

---
> **Execution order**
> 1. Cells 1–4: install, import, authenticate GEE
> 2. Cell 5 *(optional)*: verify FAO GAUL district name spelling
> 3. Cells 6–14: load all helper functions
> 4. Cell 15: display interactive widget UI → select State / District → click **▶ Run**
> 5. Cell 16: export results to Google Drive
> 6. Cell 17: ROC curve (post-analysis)
> 7. Cell 18: batch runner (optional, for multi-district production runs)


## 📦 Cell 1 — Install Dependencies

In [ ]:
# Run once per Colab session
!pip install earthengine-api geemap ipywidgets matplotlib pandas numpy scipy --quiet
print("✅ Packages installed.")


## 📚 Cell 2 — Imports

In [ ]:
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from scipy import stats
import math
import warnings
warnings.filterwarnings('ignore')
print("✅ All libraries imported.")


## 🔑 Cell 3 — GEE Authentication & Initialisation

In [ ]:
GEE_PROJECT_ID = 'your-gee-project-id'   # ← replace with your Cloud project ID

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print("✅ Earth Engine initialised.")
except Exception as e:
    print(f"Authenticating... ({e})")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print("✅ Earth Engine authenticated and initialised.")


## 🗺️ Cell 4 — West Coast State–District Configuration

Maps display names → FAO GAUL ADM2_NAME + default map centre.
Run Cell 5 to verify exact GAUL spellings for any state before the first run.


In [ ]:
DISTRICT_CONFIG = {
    "Gujarat": {
        "Kachchh (Kutch)": {"gaul":"Kachchh",     "center":[23.7,70.2],"zoom":9},
        "Porbandar":        {"gaul":"Porbandar",    "center":[21.6,69.6],"zoom":10},
        "Jamnagar":         {"gaul":"Jamnagar",     "center":[22.5,70.1],"zoom":10},
        "Junagadh":         {"gaul":"Junagadh",     "center":[21.5,70.5],"zoom":10},
        "Amreli":           {"gaul":"Amreli",       "center":[21.6,71.2],"zoom":10},
        "Bhavnagar":        {"gaul":"Bhavnagar",    "center":[21.8,72.2],"zoom":10},
        "Bharuch":          {"gaul":"Bharuch",      "center":[21.7,73.0],"zoom":10},
        "Surat":            {"gaul":"Surat",        "center":[21.2,72.8],"zoom":10},
        "Navsari":          {"gaul":"Navsari",      "center":[20.9,72.9],"zoom":11},
        "Valsad":           {"gaul":"Valsad",       "center":[20.6,72.9],"zoom":11},
    },
    "Maharashtra": {
        "Palghar":          {"gaul":"Thane",           "center":[19.7,72.8],"zoom":10},
        "Thane":            {"gaul":"Thane",           "center":[19.2,73.0],"zoom":10},
        "Mumbai City":      {"gaul":"Mumbai",          "center":[18.9,72.8],"zoom":12},
        "Mumbai Suburban":  {"gaul":"Mumbai Suburban", "center":[19.1,72.9],"zoom":11},
        "Raigad":           {"gaul":"Raigad",          "center":[18.5,73.2],"zoom":10},
        "Ratnagiri":        {"gaul":"Ratnagiri",       "center":[17.0,73.5],"zoom":10},
        "Sindhudurg":       {"gaul":"Sindhudurg",      "center":[16.0,73.8],"zoom":10},
    },
    "Goa": {
        "North Goa":        {"gaul":"North Goa",   "center":[15.5,73.9],"zoom":11},
        "South Goa":        {"gaul":"South Goa",   "center":[15.1,74.1],"zoom":11},
    },
    "Karnataka": {
        "Uttara Kannada":   {"gaul":"Uttar Kannand",  "center":[14.8,74.7],"zoom":10},
        "Udupi":            {"gaul":"Udupi",           "center":[13.3,74.8],"zoom":11},
        "Dakshina Kannada": {"gaul":"Dakshin Kannad",  "center":[12.9,75.2],"zoom":10},
    },
    "Kerala": {
        "Kasaragod":              {"gaul":"Kasaragod",          "center":[12.5,75.0],"zoom":10},
        "Kannur":                 {"gaul":"Kannur",             "center":[11.9,75.4],"zoom":10},
        "Kozhikode":              {"gaul":"Kozhikode",          "center":[11.2,75.8],"zoom":10},
        "Malappuram":             {"gaul":"Malappuram",         "center":[11.0,76.1],"zoom":10},
        "Thrissur":               {"gaul":"Thrissur",           "center":[10.5,76.2],"zoom":10},
        "Ernakulam":              {"gaul":"Ernakulam",          "center":[10.0,76.3],"zoom":10},
        "Alappuzha (Alleppey)":   {"gaul":"Alappuzha",          "center":[9.5, 76.4],"zoom":10},
        "Kollam":                 {"gaul":"Kollam",             "center":[8.9, 76.6],"zoom":10},
        "Thiruvananthapuram":     {"gaul":"Thiruvananthapuram", "center":[8.5, 76.9],"zoom":10},
    },
    "Tamil Nadu": {
        "Kanyakumari":      {"gaul":"Kanyakumari", "center":[8.1,77.3],"zoom":11},
    },
}

total = sum(len(v) for v in DISTRICT_CONFIG.values())
print(f"✅ {total} coastal districts loaded across {len(DISTRICT_CONFIG)} states.")
print("   " + " · ".join(DISTRICT_CONFIG.keys()))


## 🔎 Cell 5 — FAO GAUL Name Verifier *(Optional)*

Uncomment one line to print exact ADM2 district spellings for any state.
Update `DISTRICT_CONFIG` in Cell 4 if a name mismatches.


In [ ]:
def list_gaul_districts(state_name: str) -> list:
    """Return and print all FAO GAUL ADM2_NAME values for an Indian state."""
    fc    = (ee.FeatureCollection('FAO/GAUL/2015/level2')
              .filter(ee.Filter.eq('ADM1_NAME', state_name)))
    n     = fc.size().getInfo()
    if n == 0:
        all_s = (ee.FeatureCollection('FAO/GAUL/2015/level1')
                  .filter(ee.Filter.eq('ADM0_NAME','India'))
                  .aggregate_array('ADM1_NAME').getInfo())
        hits  = [s for s in all_s if state_name[:4].lower() in s.lower()]
        print(f"⚠️  No result for '{state_name}'. Close matches: {hits}")
        return []
    names = sorted(fc.aggregate_array('ADM2_NAME').getInfo())
    print(f"\n📍 GAUL ADM2 names for '{state_name}' ({n} districts):")
    for nm in names:
        print(f"   '{nm}'")
    return names

# ── Uncomment to verify ──────────────────────────────────────────
# list_gaul_districts("Gujarat")
# list_gaul_districts("Maharashtra")
# list_gaul_districts("Goa")
# list_gaul_districts("Karnataka")
# list_gaul_districts("Kerala")
# list_gaul_districts("Tamil Nadu")
print("💡 Uncomment a line above and run to verify GAUL names.")


## ⚙️ Cell 6 — Conditioning Factor Builder (17 bands)

**SAR bands are deliberately excluded here.**
Sentinel-1 VV data is used only inside `build_flood_inventory()` to generate the binary
flood mask for training point sampling.  Including SAR as a predictor after deriving
flood/non-flood labels from SAR creates circular data leakage and inflates accuracy metrics.

The 17 retained factors cover terrain, hydrology, climate, vegetation, soil, and land cover.


In [ ]:
# ── Band list (order maintained throughout the pipeline) ─────────────
BAND_NAMES_17 = [
    'Elevation','Slope','Aspect','TPI','TWI','Curvature',
    'SPI','FlowAcc','RiverDist',
    'Precipitation','NDVI','NDWI','LULC',
    'SoilMoisture','Clay','Lithology','RoadDist'
]

def build_conditioning_factors(roi: ee.Geometry) -> ee.Image:
    """
    Build 17-band predictor image clipped to the ROI.

    SAR_Change and SAR_FloodFreq are intentionally excluded:
    both bands are derived from Sentinel-1 VV, which is also the source
    for the flood inventory labels — including them as predictors would be
    circular and artificially inflate model accuracy.

    Returns
    -------
    ee.Image  17 named bands, unmasked (unmask(0)) and clipped to roi.
    """
    # ── Landsat 8 SR (2020-2023, <20% cloud) → NDVI, NDWI ──────────────
    landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
               .filterBounds(roi)
               .filterDate('2020-01-01','2023-12-31')
               .filter(ee.Filter.lt('CLOUD_COVER', 20))
               .map(lambda img: (img
                   .select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
                   .multiply(0.0000275).add(-0.2)
                   .copyProperties(img,['system:time_start'])))
               .median().clip(roi))
    ndvi = landsat.normalizedDifference(['SR_B5','SR_B4']).unmask(0).rename('NDVI')
    ndwi = landsat.normalizedDifference(['SR_B3','SR_B5']).unmask(0).rename('NDWI')

    # ── Terrain (SRTM 30 m) ─────────────────────────────────────────────
    dem       = ee.Image('USGS/SRTMGL1_003').clip(roi).unmask(0).rename('Elevation')
    slope     = ee.Terrain.slope(dem).rename('Slope')
    aspect    = ee.Terrain.aspect(dem).rename('Aspect')
    curvature = dem.convolve(ee.Kernel.laplacian8()).unmask(0).rename('Curvature')
    tpi       = (dem.subtract(dem.focal_mean(300,'circle','meters'))
                 .unmask(0).clip(roi).rename('TPI'))
    slope_rad = slope.multiply(math.pi / 180)
    twi       = (ee.Image.pixelArea().multiply(1e6)
                 .divide(slope_rad.tan().add(0.001))
                 .log().unmask(0).clip(roi).rename('TWI'))

    # ── Hydrological indices ─────────────────────────────────────────────
    flow_acc = ee.Image('WWF/HydroSHEDS/15ACC').clip(roi).rename('FlowAcc')
    spi      = (flow_acc.multiply(ee.Image.pixelArea())
                .multiply(slope_rad.tan().add(0.001))
                .log().unmask(0).clip(roi).rename('SPI'))

    # Distance to rivers — HydroSHEDS (global; avoids US-only TIGER issue)
    rivers     = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers')
    river_rast = (ee.Image().byte()
                  .paint(rivers.filterBounds(roi), 1)
                  .selfMask())
    river_dist = (river_rast.fastDistanceTransform(512).sqrt()
                  .multiply(ee.Image.pixelArea().sqrt())
                  .unmask(0).clip(roi).rename('RiverDist'))

    # ── Land cover, soil, geology, accessibility ─────────────────────────
    lulc      = (ee.ImageCollection('ESA/WorldCover/v200').first()
                 .clip(roi).unmask(0).rename('LULC'))
    soil_mois = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                 .filterBounds(roi).filterDate('2018-01-01','2023-12-31')
                 .filter(ee.Filter.calendarRange(6,9,'month'))
                 .select('volumetric_soil_water_layer_1')
                 .mean().unmask(0).clip(roi).rename('SoilMoisture'))
    clay      = (ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02')
                 .select('b0').clip(roi).unmask(0).rename('Clay'))
    lithology = (ee.Image('CSP/ERGo/1_0/Global/SRTM_landforms')
                 .clip(roi).unmask(0).rename('Lithology'))
    road_dist = (ee.Image('Oxford/MAP/accessibility_to_cities_2015_v1_0')
                 .select('accessibility').unmask(0).clip(roi).rename('RoadDist'))

    # ── CHIRPS monsoon precipitation (mean annual JJAS 2018-2023) ────────
    def yr_rain(y):
        y = ee.Number(y)
        return (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .filterBounds(roi)
                .filter(ee.Filter.calendarRange(y,y,'year'))
                .filter(ee.Filter.calendarRange(6,9,'month'))
                .sum().clip(roi))
    chirps = (ee.ImageCollection(ee.List.sequence(2018,2023).map(yr_rain))
              .mean().unmask(0).rename('Precipitation'))

    # ── Stack 17 predictors (SAR intentionally omitted) ─────────────────
    stack = (dem
        .addBands(slope).addBands(aspect).addBands(tpi).addBands(twi)
        .addBands(curvature).addBands(spi).addBands(flow_acc)
        .addBands(river_dist).addBands(chirps)
        .addBands(ndvi).addBands(ndwi).addBands(lulc)
        .addBands(soil_mois).addBands(clay).addBands(lithology).addBands(road_dist))

    assert stack.bandNames().size().getInfo() == 17, "Band count mismatch!"
    print(f"  [build_conditioning_factors] 17 bands ready (SAR excluded).")
    return stack

print("✅ build_conditioning_factors() defined  — 17 bands, SAR-free.")


## 🌊 Cell 7 — Flood Inventory Builder

SAR is used **here and only here** — to derive the binary flood mask used for
sampling training points (GFD union with Sentinel-1 VV < −15 dB).
Once training labels are created, SAR plays no further role in the modelling pipeline.


In [ ]:
def build_flood_inventory(
    roi: ee.Geometry,
    n_flood:     int = 500,
    n_non_flood: int = 500,
    seed:        int = 42,
) -> tuple:
    """
    Construct flood / non-flood sample points from multi-source fusion.

    Sources fused (logical OR):
      • Global Flood Database — MODIS 500 m, events ~2000-2018
      • Sentinel-1 VV < −15 dB during monsoon months (June-October), 2018-2023

    Returns
    -------
    flood_points, non_flood_points : ee.FeatureCollection (each with 'label' property)
    combined_flood_mask            : ee.Image (binary, 1 = flooded)
    """
    gfd = (ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
           .filterBounds(roi).select('flooded').max().clip(roi))

    sar_flood = (ee.ImageCollection('COPERNICUS/S1_GRD')
                 .filterBounds(roi).filterDate('2018-06-01','2023-10-31')
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV'))
                 .filter(ee.Filter.eq('instrumentMode','IW'))
                 .filter(ee.Filter.calendarRange(6,10,'month'))
                 .select('VV')
                 .map(lambda img: img.lt(-15).rename('flooded'))
                 .max().clip(roi))

    union         = gfd.gt(0).unmask(0).Or(sar_flood.gt(0).unmask(0))
    flood_mask    = union.rename('flooded').selfMask()
    non_flood_mask= union.eq(0).rename('flooded').selfMask()

    fp  = (flood_mask.stratifiedSample(
               numPoints=n_flood, classBand='flooded', region=roi,
               scale=100, seed=seed, geometries=True)
           .map(lambda f: f.set('label',1)))
    nfp = (non_flood_mask.stratifiedSample(
               numPoints=n_non_flood, classBand='flooded', region=roi,
               scale=100, seed=seed, geometries=True)
           .map(lambda f: f.set('label',0)))

    print(f"  [build_flood_inventory] "
          f"Flood: {fp.size().getInfo()} pts | Non-flood: {nfp.size().getInfo()} pts")
    return fp, nfp, flood_mask

print("✅ build_flood_inventory() defined.")


## 📊 Cell 8 — Variable Selection & Three-Way Data Split

**Why three splits?**  
Conformal prediction requires a *held-out calibration set* that is independent of both
training and final evaluation.  The split is:

| Partition | Fraction | Purpose |
|-----------|----------|---------|
| **Train** | 60 % | RF model fitting + variable importance ranking |
| **Calibration** | 20 % | Nonconformity score estimation for conformal prediction |
| **Test** | 20 % | Accuracy assessment + empirical coverage validation |

An initial 200-tree RF is trained on the training split to rank variable importance;
only the top-N predictors (user-defined) are passed to the final RF.


In [ ]:
def prepare_samples(
    all_predictors: ee.Image,
    flood_pts:      ee.FeatureCollection,
    non_flood_pts:  ee.FeatureCollection,
    top_n:          int,
    train_frac:     float = 0.60,
    cal_frac:       float = 0.20,
) -> tuple:
    """
    Sample all 17 predictors at flood/non-flood points, rank by RF importance,
    select top-N, and return three independent data partitions.

    Parameters
    ----------
    all_predictors : ee.Image  — full 17-band predictor stack
    flood_pts      : ee.FeatureCollection
    non_flood_pts  : ee.FeatureCollection
    top_n          : int  — number of top-importance variables to retain
    train_frac     : float  — fraction for training (default 0.60)
    cal_frac       : float  — fraction for calibration (default 0.20)
                     remainder → test set

    Returns
    -------
    selected_predictors : ee.Image  (top-N bands only)
    train_set_sel       : ee.FeatureCollection  (60%, top-N + label)
    cal_set_sel         : ee.FeatureCollection  (20%, top-N + label)
    test_set_sel        : ee.FeatureCollection  (20%, top-N + label)
    selected_vars       : ee.List  of selected band name strings
    imp_dict            : ee.Dictionary  {band: importance}
    """
    cal_upper = train_frac + cal_frac           # e.g. 0.80

    all_bands   = all_predictors.bandNames()
    samples     = flood_pts.merge(non_flood_pts)
    sampled_raw = all_predictors.sampleRegions(
                      collection=samples, properties=['label'], scale=30)

    split      = sampled_raw.randomColumn('rnd', 42)
    train_set  = split.filter(ee.Filter.lt('rnd', train_frac))
    cal_set    = split.filter(ee.Filter.And(
                     ee.Filter.gte('rnd', train_frac),
                     ee.Filter.lt('rnd', cal_upper)))
    test_set   = split.filter(ee.Filter.gte('rnd', cal_upper))

    n_tr = train_set.size().getInfo()
    n_ca = cal_set.size().getInfo()
    n_te = test_set.size().getInfo()
    print(f"  [prepare_samples] Train: {n_tr} | Cal: {n_ca} | Test: {n_te}")

    # ── Initial RF on ALL 17 bands for importance ranking ─────────────────
    rf_init  = (ee.Classifier.smileRandomForest(numberOfTrees=200, seed=42)
                .train(train_set,'label',all_bands))
    imp_dict = ee.Dictionary(rf_init.explain().get('importance'))

    imp_fc = ee.FeatureCollection(all_bands.map(lambda b: ee.Feature(None, {
        'Variable':   b,
        'Importance': ee.Number(imp_dict.get(b)).max(0)
    }))).sort('Importance', False)

    sel_fc   = ee.FeatureCollection(imp_fc.toList(top_n))
    sel_vars = sel_fc.aggregate_array('Variable')

    sel_preds     = all_predictors.select(sel_vars)
    band_plus_lbl = sel_vars.add('label')
    train_sel     = train_set.select(band_plus_lbl)
    cal_sel       = cal_set.select(band_plus_lbl)
    test_sel      = test_set.select(band_plus_lbl)

    print(f"  [prepare_samples] Top-{top_n} selected: {sel_vars.getInfo()}")
    return sel_preds, train_sel, cal_sel, test_sel, sel_vars, imp_dict

print("✅ prepare_samples() defined  — 3-way split (train/cal/test).")


## 🤖 Cell 9 — Random Forest Training

Trains the final RF in two output modes:
- **PROBABILITY** → susceptibility map and all UQ layers
- **CLASSIFICATION** → confusion-matrix accuracy assessment

The same hyperparameter dict feeds both classifiers, ensuring consistency.


In [ ]:
def get_rf_params(preset: str, widget_refs: dict) -> dict:
    """
    Return RF hyperparameter dict based on preset or custom widget values.

    Presets
    -------
    Default : GEE defaults (good baseline for any region)
    Tuned   : Validated settings for coastal Indian districts
    Custom  : Read from ipywidgets
    """
    if preset == 'Default':
        return dict(numberOfTrees=500, variablesPerSplit=None,
                    minLeafPopulation=1, bagFraction=0.5, seed=42)
    elif preset == 'Tuned':
        return dict(numberOfTrees=500, variablesPerSplit=4,
                    minLeafPopulation=5, bagFraction=0.6, seed=42)
    else:
        return dict(
            numberOfTrees     = int(widget_refs['rf_trees'].value),
            variablesPerSplit = int(widget_refs['rf_varsplit'].value),
            minLeafPopulation = int(widget_refs['rf_minleaf'].value),
            bagFraction       = float(widget_refs['rf_bag'].value),
            seed=42,
        )


def train_rf(train_sel: ee.FeatureCollection,
              band_names: ee.List,
              rf_params: dict) -> dict:
    """
    Train a Random Forest classifier in both PROBABILITY and CLASSIFICATION modes.

    Returns
    -------
    dict with keys:
        'prob' : ee.Classifier  — output mode PROBABILITY
        'hard' : ee.Classifier  — output mode CLASSIFICATION
    """
    rf_prob = (ee.Classifier.smileRandomForest(**rf_params)
               .setOutputMode('PROBABILITY')
               .train(train_sel,'label',band_names))
    rf_hard = (ee.Classifier.smileRandomForest(**rf_params)
               .train(train_sel,'label',band_names))

    print(f"  [train_rf] RF trained — trees={rf_params['numberOfTrees']}, "
          f"bagFrac={rf_params['bagFraction']}, "
          f"varsPerSplit={rf_params['variablesPerSplit']}")
    return {'prob': rf_prob, 'hard': rf_hard}

print("✅ get_rf_params() and train_rf() defined.")


## 🗺️ Cell 10 — RF Susceptibility Map & 5-Class FSM

Generates the primary flood susceptibility probability map and reclassifies it into
five ordinal classes using equal-interval thresholds (0.2 step) matching the standard
FAO/UNDRR flood susceptibility classification scheme.

| Class | Label | Probability range |
|-------|-------|-------------------|
| 1 | Very Low | 0.0 – 0.2 |
| 2 | Low | 0.2 – 0.4 |
| 3 | Moderate | 0.4 – 0.6 |
| 4 | High | 0.6 – 0.8 |
| 5 | Very High | 0.8 – 1.0 |


In [ ]:
FSM_PALETTE = ['#1a9641','#a6d96a','#ffffbf','#fdae61','#d7191c']

def compute_fsm(rf_classifiers: dict,
                selected_predictors: ee.Image) -> tuple:
    """
    Apply the trained RF probability classifier to the full district image.

    Returns
    -------
    rf_prob  : ee.Image  — continuous probability [0,1]
    rf_class : ee.Image  — 5-class FSM (byte, values 1-5)
    """
    rf_prob = (selected_predictors
               .classify(rf_classifiers['prob'])
               .rename('RF_Prob'))

    rf_class = (rf_prob
                .where(rf_prob.lte(0.2), 1)
                .where(rf_prob.gt(0.2).And(rf_prob.lte(0.4)), 2)
                .where(rf_prob.gt(0.4).And(rf_prob.lte(0.6)), 3)
                .where(rf_prob.gt(0.6).And(rf_prob.lte(0.8)), 4)
                .where(rf_prob.gt(0.8), 5)
                .rename('RF_Class').toByte())

    print("  [compute_fsm] RF probability map and 5-class FSM ready.")
    return rf_prob, rf_class

print("✅ compute_fsm() defined.")


## 🔬 Cell 11 — Four-Method Uncertainty Quantification

### UQ-1  Bernoulli Variance  `p(1−p)`
The variance of a Bernoulli(p) variable is maximised at p = 0.5 (decision boundary) and
falls to zero at p ∈ {0,1} (certain prediction). In hydrology FSM literature this is the
most widely reported within-model uncertainty indicator (Tehrany et al., 2019; Wang et al., 2020).

### UQ-2  Margin Confidence  `min(p, 1−p)`
Measures how far the predicted probability is from the 0.5 classification boundary.
Pixels with values near 0.5 are the least confident — often spatially correlated with
transition zones between flood-prone and upland areas.

### UQ-3  Bootstrap RF Variance
K Random Forest models are trained on independent sub-samples of the training data
(drawn without replacement, 80% fraction, different seeds). The pixel-wise standard
deviation of K probability maps quantifies **parameter / epistemic uncertainty** — how
sensitive predictions are to the specific sample of flood records used for training.
This is the standard bootstrap estimator used in hydrology uncertainty analysis (e.g.
GLUE, SWAT-based inference frameworks).

### UQ-4  Inductive Conformal Prediction
**Distribution-free** coverage guarantee at user-specified significance level α:
the resulting prediction sets contain the true class with empirical frequency ≥ 1−α,
regardless of the RF model's internal calibration. Implemented via the split-conformal
(inductive) procedure (Papadopoulos et al., 2002; Angelopoulos & Bates, 2021):

1. Compute nonconformity scores on the held-out calibration set:
   `A_i = 1 − P̂(y_i | x_i)` where `y_i` is the true class label.
2. Set threshold: `q = Quantile(A_cal, ⌈(n_cal+1)(1−α)⌉ / n_cal)`.
3. At each image pixel assign a **prediction set**:
   - *Flood* included if `1 − P̂(flood | x) ≤ q`, i.e. `P̂(flood|x) ≥ 1 − q`
   - *Non-flood* included if `P̂(flood | x) ≤ q`
4. **Conformal uncertainty zone**: pixels where the prediction set is ambiguous
   (`{flood, non-flood}` both included, or empty — anomaly).
5. **Empirical coverage** on the independent test set validates the guarantee.


In [ ]:
def compute_uq_all(
    rf_classifiers:     dict,
    selected_predictors:ee.Image,
    train_sel:          ee.FeatureCollection,
    cal_sel:            ee.FeatureCollection,
    test_sel:           ee.FeatureCollection,
    band_names:         ee.List,
    rf_params:          dict,
    n_bootstrap:        int   = 5,
    alpha:              float = 0.10,
) -> dict:
    """
    Compute all four UQ layers and return them together with conformal diagnostics.

    Parameters
    ----------
    rf_classifiers      : output of train_rf()
    selected_predictors : top-N predictor image
    train_sel           : training FeatureCollection (60%)
    cal_sel             : calibration FeatureCollection (20%)
    test_sel            : test FeatureCollection (20%)
    band_names          : ee.List of predictor band names
    rf_params           : dict from get_rf_params()
    n_bootstrap         : number of bootstrap RF models (default 5)
    alpha               : conformal significance level (default 0.10 → 90% coverage)

    Returns
    -------
    dict with keys:
        uq_variance, uq_margin, uq_bootstrap          : ee.Image (continuous)
        conformal_pred_set                             : ee.Image (categorical 0-3)
        conformal_uncertain_zone                       : ee.Image (binary)
        q_threshold                                    : float
        cal_nc_scores                                  : list[float]
        test_nc_scores                                 : list[float]
        empirical_coverage                             : float
    """
    rf_prob_img = selected_predictors.classify(rf_classifiers['prob'])

    # ── UQ-1: Bernoulli Variance  p(1-p) ────────────────────────────────
    p = rf_prob_img.max(1e-7).min(1 - 1e-7)
    uq_variance = p.multiply(ee.Image(1).subtract(p)).rename('UQ_Variance')
    print("  [UQ-1] Bernoulli variance computed.")

    # ── UQ-2: Margin Confidence  min(p, 1-p) ────────────────────────────
    uq_margin = p.min(ee.Image(1).subtract(p)).rename('UQ_Margin')
    print("  [UQ-2] Margin confidence computed.")

    # ── UQ-3: Bootstrap RF Variance ──────────────────────────────────────
    print(f"  [UQ-3] Training {n_bootstrap} bootstrap RF models...")
    boot_prob_images = []
    for k in range(n_bootstrap):
        # Draw different 80% sub-sample of training data via new random column
        boot_train = (train_sel
                      .randomColumn(f'boot_{k}', seed=k*17+3)
                      .filter(ee.Filter.lt(f'boot_{k}', 0.80)))
        boot_rf = (ee.Classifier.smileRandomForest(**{**rf_params, 'seed': k})
                   .setOutputMode('PROBABILITY')
                   .train(boot_train,'label',band_names))
        boot_prob_images.append(
            selected_predictors.classify(boot_rf).rename(f'boot_{k}'))

    boot_stack   = boot_prob_images[0]
    for img in boot_prob_images[1:]:
        boot_stack = boot_stack.addBands(img)

    boot_mean  = boot_stack.reduce(ee.Reducer.mean())
    boot_var   = boot_stack.subtract(boot_mean).pow(2).reduce(ee.Reducer.mean())
    uq_bootstrap = boot_var.sqrt().rename('UQ_Bootstrap')
    print(f"  [UQ-3] Bootstrap variance computed ({n_bootstrap} models).")

    # ── UQ-4: Inductive Conformal Prediction ─────────────────────────────
    print(f"  [UQ-4] Computing conformal prediction (α={alpha})...")

    #  Step A — nonconformity scores on calibration set
    cal_classified = cal_sel.classify(rf_classifiers['prob'])
    cal_with_nc = cal_classified.map(lambda f: f.set(
        'nc_score',
        ee.Algorithms.If(
            ee.Number(f.get('label')).eq(1),
            ee.Number(1).subtract(f.get('classification')),   # flood:     1 − p̂_flood
            f.get('classification')                            # non-flood: p̂_flood
        )
    ))
    cal_nc_scores = cal_with_nc.aggregate_array('nc_score').getInfo()
    n_cal         = len(cal_nc_scores)

    # Step B — empirical (1−α) quantile with finite-sample correction
    # q = quantile at level ceil((n_cal+1)(1-α)) / n_cal
    q_level      = min(1.0, math.ceil((n_cal + 1) * (1 - alpha)) / n_cal)
    q_threshold  = float(np.quantile(cal_nc_scores, q_level))
    print(f"  [UQ-4] Calibration: n_cal={n_cal}, q_{1-alpha:.0%}={q_threshold:.4f}")

    # Step C — prediction set image
    # Include flood     if  1−p ≤ q  ↔  p ≥ 1−q
    # Include non-flood if  p   ≤ q
    pred_flood_img    = rf_prob_img.gte(1.0 - q_threshold)   # 1 = flood in set
    pred_nonflood_img = rf_prob_img.lte(q_threshold)          # 1 = non-flood in set
    # Encoding: flood contributes bit-1 (×2), non-flood contributes bit-0 (×1)
    # Result: 0=empty(anomaly), 1=non-flood only, 2=flood only, 3=both(uncertain)
    conformal_pred_set = (pred_flood_img.multiply(2)
                          .add(pred_nonflood_img)
                          .rename('Conformal_PredSet').toByte())

    # Conformal uncertain zone = prediction set is NOT a singleton
    conformal_uncertain_zone = (conformal_pred_set.neq(1).And(conformal_pred_set.neq(2))
                                .rename('Conformal_UncertainZone').toByte())

    # Step D — nonconformity scores on test set + empirical coverage
    test_classified = test_sel.classify(rf_classifiers['prob'])
    test_with_nc = test_classified.map(lambda f: f.set(
        'nc_score',
        ee.Algorithms.If(
            ee.Number(f.get('label')).eq(1),
            ee.Number(1).subtract(f.get('classification')),
            f.get('classification')
        )
    ))
    test_nc_scores     = test_with_nc.aggregate_array('nc_score').getInfo()
    empirical_coverage = float(np.mean(np.array(test_nc_scores) <= q_threshold))
    nominal_coverage   = 1 - alpha
    status = "✅ VALID" if empirical_coverage >= nominal_coverage - 0.005 else "⚠️ BELOW TARGET"
    print(f"  [UQ-4] Empirical coverage = {empirical_coverage:.4f}  "
          f"(target ≥ {nominal_coverage:.2f})  {status}")

    return {
        'uq_variance':           uq_variance,
        'uq_margin':             uq_margin,
        'uq_bootstrap':          uq_bootstrap,
        'conformal_pred_set':    conformal_pred_set,
        'conformal_uncertain_zone': conformal_uncertain_zone,
        'q_threshold':           q_threshold,
        'cal_nc_scores':         cal_nc_scores,
        'test_nc_scores':        test_nc_scores,
        'empirical_coverage':    empirical_coverage,
        'n_bootstrap':           n_bootstrap,
        'alpha':                 alpha,
    }

print("✅ compute_uq_all() defined  — UQ-1 Bernoulli Var · UQ-2 Margin · UQ-3 Bootstrap · UQ-4 Conformal.")


## 📋 Cell 12 — RF Accuracy Assessment

Evaluates the hard RF classifier on the 20% test split.
Accuracy, Sensitivity (Recall), Specificity, Precision and F1 are derived
from the binary confusion matrix.


In [ ]:
def assess_accuracy_rf(rf_classifiers: dict,
                        test_sel:      ee.FeatureCollection) -> pd.DataFrame:
    """
    Evaluate the hard RF classifier on the test partition.

    Returns a single-row DataFrame with columns:
        Accuracy, Sensitivity, Specificity, Precision, F1_Score, Kappa
    """
    validated = test_sel.classify(rf_classifiers['hard'])
    cm        = validated.errorMatrix('label','classification')
    cm_arr    = cm.array()

    tn = ee.Number(ee.List(ee.List(cm_arr.toList()).get(0)).get(0))
    fp = ee.Number(ee.List(ee.List(cm_arr.toList()).get(0)).get(1))
    fn = ee.Number(ee.List(ee.List(cm_arr.toList()).get(1)).get(0))
    tp = ee.Number(ee.List(ee.List(cm_arr.toList()).get(1)).get(1))

    n         = tp.add(tn).add(fp).add(fn)
    accuracy  = tp.add(tn).divide(n)
    sens      = tp.divide(tp.add(fn))
    spec      = tn.divide(tn.add(fp))
    prec      = tp.divide(tp.add(fp))
    f1        = prec.multiply(sens).multiply(2).divide(prec.add(sens))
    # Cohen's Kappa
    p_e       = (tp.add(fp).divide(n)).multiply(tp.add(fn).divide(n)).add(
                 (tn.add(fn).divide(n)).multiply(tn.add(fp).divide(n)))
    kappa     = accuracy.subtract(p_e).divide(ee.Number(1).subtract(p_e))

    metrics = ee.Dictionary({
        'Accuracy': accuracy, 'Sensitivity': sens, 'Specificity': spec,
        'Precision': prec, 'F1_Score': f1, 'Kappa': kappa,
    }).getInfo()

    df = pd.DataFrame([{k: round(v,4) if v is not None else 'N/A'
                         for k,v in metrics.items()}],
                       index=['Random Forest'])
    df.index.name = 'Model'
    return df


def print_accuracy_table(df: pd.DataFrame, district_name: str):
    print(f"\n{'─'*58}")
    print(f"  📊 Accuracy — {district_name}")
    print(f"{'─'*58}")
    print(df.to_string())
    print(f"{'─'*58}\n")

print("✅ assess_accuracy_rf() defined.")


## 📈 Cell 13 — Plotting Functions

- **Variable importance** — horizontal bar chart (top-N, sorted descending)
- **UQ comparison** — 2×2 histogram grid for all 4 UQ layers
- **UQ correlation matrix** — pairwise scatter + Pearson r for UQ-1/2/3
- **Conformal calibration curve** — empirical vs nominal coverage across α levels
- **Conformal prediction-set frequency** — bar chart of class 0/1/2/3 pixel counts
- **Area distribution** — percent area per FSM class


In [ ]:
UQ_PALETTE  = ['#ffffcc','#fd8d3c','#bd0026']
CONF_PALETTE= ['#2166ac','#4dac26','#d01c8b','#f1b6da']  # 0=anomaly,1=nonflood,2=flood,3=uncertain

def plot_variable_importance(imp_dict: ee.Dictionary,
                              sel_vars: ee.List,
                              district_name: str):
    """Horizontal bar chart of selected variable importance (sorted descending)."""
    raw      = imp_dict.getInfo()
    sel      = sel_vars.getInfo()
    pairs    = sorted([(v, raw.get(v, 0)) for v in sel], key=lambda x: x[1])
    names, scores = zip(*pairs)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'RF Variable Importance — {district_name}', fontweight='bold', fontsize=13)

    # All 17 (left)
    all_pairs = sorted(raw.items(), key=lambda x: x[1], reverse=True)
    all_n, all_s = zip(*all_pairs)
    axes[0].barh(all_n, all_s, color='#1565c0', edgecolor='white')
    axes[0].set_title('All 17 Candidates')
    axes[0].set_xlabel('Importance Score')
    axes[0].invert_yaxis()

    # Selected top-N (right)
    axes[1].barh(names, scores, color='#d32f2f', edgecolor='white')
    axes[1].set_title(f'Selected Top-{len(sel)} (descending)')
    axes[1].set_xlabel('Importance Score')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig('/content/variable_importance.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_uq_histograms(uq_results: dict, roi: ee.Geometry,
                        rf_prob: ee.Image, district_name: str):
    """2×2 histogram grid for the four UQ layers sampled at 1000 pixels."""
    print("  [plot_uq_histograms] Sampling UQ layers (1000 px, ~30 s)...")
    stack = (uq_results['uq_variance'].rename('UQ1')
             .addBands(uq_results['uq_margin'].rename('UQ2'))
             .addBands(uq_results['uq_bootstrap'].rename('UQ3'))
             .addBands(rf_prob.rename('RF_Prob')))
    raw   = stack.sample(region=roi, scale=500, numPixels=1000,
                          seed=99, geometries=False).getInfo()['features']
    if not raw:
        print("  ⚠️  No samples returned — increase scale or check ROI extent.")
        return
    df = pd.DataFrame([f['properties'] for f in raw])

    # Conformal pred-set pixel counts (client-side from image stats)
    conf_hist = uq_results['conformal_pred_set'].reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=roi, scale=500, bestEffort=True, maxPixels=1e8
    ).get('Conformal_PredSet').getInfo() or {}

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f'Uncertainty Quantification — {district_name}', fontweight='bold', fontsize=14)
    gs  = gridspec.GridSpec(2, 3, figure=fig)

    ax1 = fig.add_subplot(gs[0,0])
    ax2 = fig.add_subplot(gs[0,1])
    ax3 = fig.add_subplot(gs[0,2])
    ax4 = fig.add_subplot(gs[1,0])
    ax5 = fig.add_subplot(gs[1,1])
    ax6 = fig.add_subplot(gs[1,2])

    # UQ-1 Bernoulli Variance
    ax1.hist(df['UQ1'].dropna(), bins=40, color='#1565c0', edgecolor='white', alpha=0.85)
    ax1.set_title('UQ-1: Bernoulli Variance p(1−p)', fontsize=11)
    ax1.set_xlabel('Variance'); ax1.set_ylabel('Count')
    ax1.axvline(df['UQ1'].mean(), color='red', ls='--',
                label=f'μ={df["UQ1"].mean():.4f}')
    ax1.legend(fontsize=9)

    # UQ-2 Margin Confidence
    ax2.hist(df['UQ2'].dropna(), bins=40, color='#388e3c', edgecolor='white', alpha=0.85)
    ax2.set_title('UQ-2: Margin Confidence min(p,1−p)', fontsize=11)
    ax2.set_xlabel('Margin uncertainty')
    ax2.axvline(df['UQ2'].mean(), color='red', ls='--',
                label=f'μ={df["UQ2"].mean():.4f}')
    ax2.legend(fontsize=9)

    # UQ-3 Bootstrap Variance
    ax3.hist(df['UQ3'].dropna(), bins=40, color='#e65100', edgecolor='white', alpha=0.85)
    ax3.set_title(f'UQ-3: Bootstrap Std Dev (K={uq_results["n_bootstrap"]})', fontsize=11)
    ax3.set_xlabel('Std Dev')
    ax3.axvline(df['UQ3'].mean(), color='navy', ls='--',
                label=f'μ={df["UQ3"].mean():.4f}')
    ax3.legend(fontsize=9)

    # UQ-4 Conformal prediction set frequencies
    labels_conf = {0:'Empty\n(anomaly)', 1:'Non-flood\nonly',
                    2:'Flood\nonly',      3:'Both\n(uncertain)'}
    conf_keys   = sorted(conf_hist.keys(), key=lambda x: int(x))
    conf_vals   = [conf_hist[k] for k in conf_keys]
    conf_labels = [labels_conf.get(int(k),'?') for k in conf_keys]
    bar_colors  = [CONF_PALETTE[int(k)] for k in conf_keys]
    ax4.bar(conf_labels, conf_vals, color=bar_colors, edgecolor='white')
    ax4.set_title(f'UQ-4: Conformal Pred Set (α={uq_results["alpha"]})', fontsize=11)
    ax4.set_ylabel('Pixel count')
    q = uq_results['q_threshold']
    ax4.set_xlabel(f'q={q:.4f} | coverage={uq_results["empirical_coverage"]:.4f}')

    # Pairwise scatter: UQ-1 vs UQ-2
    ax5.scatter(df['UQ1'].dropna(), df['UQ2'].dropna(), alpha=0.25, s=6, color='#7b1fa2')
    r12, _ = stats.pearsonr(df['UQ1'].dropna(), df['UQ2'].dropna())
    ax5.set_xlabel('UQ-1 Bernoulli Variance')
    ax5.set_ylabel('UQ-2 Margin Confidence')
    ax5.set_title(f'UQ-1 vs UQ-2  (r={r12:.3f})', fontsize=11)

    # RF probability histogram
    ax6.hist(df['RF_Prob'].dropna(), bins=40, color='#546e7a', edgecolor='white', alpha=0.85)
    ax6.set_title('RF Susceptibility Probability', fontsize=11)
    ax6.set_xlabel('P(flood)')
    ax6.axvline(0.5, color='red', ls='--', lw=1.5, label='0.5 boundary')
    ax6.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('/content/uq_histograms.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print UQ summary statistics
    print(f"\n  UQ Summary (sampled from {len(df)} pixels):")
    print(f"  {'Metric':<32}  {'Mean':>8}  {'Std':>8}  {'Max':>8}")
    print(f"  {'─'*60}")
    for col, label in [('UQ1','UQ-1 Bernoulli Variance'),
                        ('UQ2','UQ-2 Margin Confidence'),
                        ('UQ3','UQ-3 Bootstrap Std Dev')]:
        s = df[col].dropna()
        print(f"  {label:<32}  {s.mean():8.4f}  {s.std():8.4f}  {s.max():8.4f}")
    print(f"  {'UQ-4 Empirical Coverage':<32}  {uq_results['empirical_coverage']:8.4f}")


def plot_conformal_calibration(cal_nc_scores: list,
                                test_nc_scores: list,
                                district_name: str):
    """
    Conformal calibration curve: empirical coverage on the test set across alpha levels.
    A well-calibrated conformal predictor should lie at or above the diagonal.
    """
    alphas_range = np.arange(0.02, 0.52, 0.02)
    emp_coverages = []
    for a in alphas_range:
        q_level = min(1.0, math.ceil((len(cal_nc_scores)+1)*(1-a)) / len(cal_nc_scores))
        q_a     = np.quantile(cal_nc_scores, q_level)
        cov     = np.mean(np.array(test_nc_scores) <= q_a)
        emp_coverages.append(cov)

    nominal = 1 - alphas_range
    fig, ax = plt.subplots(figsize=(7,6))
    ax.plot([0,1],[1,0],'k--', lw=1.2, label='Perfect calibration (diagonal)')
    ax.plot(alphas_range, emp_coverages, 'o-', color='#1565c0', lw=2,
            markersize=5, label='Empirical coverage (test set)')
    ax.fill_between(alphas_range, nominal, emp_coverages,
                    where=np.array(emp_coverages) >= nominal,
                    alpha=0.15, color='green', label='Conservative region')
    ax.fill_between(alphas_range, nominal, emp_coverages,
                    where=np.array(emp_coverages) < nominal,
                    alpha=0.15, color='red', label='Anti-conservative region')
    ax.set_xlabel('Significance level α', fontsize=12)
    ax.set_ylabel('Empirical coverage (1 − α̂)', fontsize=12)
    ax.set_title(f'Conformal Calibration Curve — {district_name}', fontweight='bold', fontsize=13)
    ax.set_xlim(0, 0.5); ax.set_ylim(0.45, 1.02)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/conformal_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_area_distribution(rf_class: ee.Image,
                            roi: ee.Geometry,
                            district_name: str):
    """Bar chart of % area per FSM class."""
    area_img = ee.Image.pixelArea().addBands(rf_class)
    grp      = (area_img.reduceRegion(
                    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
                    geometry=roi, scale=500, bestEffort=True, maxPixels=1e8)
                .get('groups').getInfo() or [])
    total   = sum(g['sum'] for g in grp)
    labels  = {1:'Very Low',2:'Low',3:'Moderate',4:'High',5:'Very High'}
    vals    = [next((g['sum']/total*100 for g in grp if g['class']==c), 0)
               for c in range(1,6)]

    fig, ax = plt.subplots(figsize=(7,5))
    bars = ax.bar([labels[c] for c in range(1,6)], vals,
                  color=FSM_PALETTE, edgecolor='white', width=0.65)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
    ax.set_title(f'FSM Area Distribution — {district_name}', fontweight='bold')
    ax.set_ylabel('% Area')
    ax.set_ylim(0, max(vals)*1.2 if vals else 100)
    plt.tight_layout()
    plt.savefig('/content/area_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

print("✅ All plotting functions defined.")


## 🌍 Cell 14 — geemap Layer Configuration

Defines vis-params for all 17 conditioning factor layers and the UQ layers,
then provides `add_all_layers()` which populates the geemap interactive map.


In [ ]:
LAYER_VIS = {
    'Elevation':    {'min':0,   'max':1000,'palette':['#313695','#4575b4','#74add1','#abd9e9','#fee090','#fdae61','#f46d43','#d73027']},
    'Slope':        {'min':0,   'max':45,  'palette':['#ffffcc','#a1dab4','#41b6c4','#225ea8']},
    'Aspect':       {'min':0,   'max':360, 'palette':['#d73027','#fc8d59','#fee08b','#d9ef8b','#91cf60','#1a9850','#d73027']},
    'TPI':          {'min':-50, 'max':50,  'palette':['#2166ac','#67a9cf','#f7f7f7','#ef8a62','#b2182b']},
    'TWI':          {'min':0,   'max':20,  'palette':['#ffffcc','#a1dab4','#41b6c4','#2c7fb8','#253494']},
    'Curvature':    {'min':-2,  'max':2,   'palette':['#d7191c','#ffffbf','#1a9641']},
    'SPI':          {'min':0,   'max':20,  'palette':['#ffffcc','#a1dab4','#41b6c4','#2c7fb8','#253494']},
    'FlowAcc':      {'min':0,   'max':10000,'palette':['#f7fbff','#6baed6','#08306b']},
    'RiverDist':    {'min':0,   'max':50000,'palette':['#08306b','#2171b5','#6baed6','#c6dbef','#f7fbff']},
    'Precipitation':{'min':0,   'max':3000,'palette':['#ffffcc','#a1dab4','#41b6c4','#225ea8']},
    'NDVI':         {'min':-0.2,'max':0.8, 'palette':['#d73027','#fee08b','#1a9850']},
    'NDWI':         {'min':-0.5,'max':0.5, 'palette':['#d73027','#ffffbf','#1a9850']},
    'LULC':         {'min':10,  'max':100, 'palette':['#006400','#ffbb22','#ffff4c','#f096ff','#fa0000','#b4b4b4','#f0f0f0','#0064c8','#0096a0','#00cf75','#fae6a0']},
    'SoilMoisture': {'min':0.1, 'max':0.5, 'palette':['#ffffcc','#a1dab4','#41b6c4','#2c7fb8','#253494']},
    'Clay':         {'min':0,   'max':60,  'palette':['#ffffcc','#d9f0a3','#addd8e','#78c679','#31a354','#006837']},
    'Lithology':    {'min':11,  'max':42,  'palette':['#aec3d4','#162103','#cc6d8f','#68ab63','#c2b34a','#e1e5db','#a3cc51','#d27c35']},
    'RoadDist':     {'min':0,   'max':300, 'palette':['#67001f','#d6604d','#fddbc7','#d1e5f0','#4393c3','#053061']},
}

UQ_VIS_VARIANCE  = {'min':0,   'max':0.25, 'palette':['#ffffcc','#fd8d3c','#bd0026']}
UQ_VIS_MARGIN    = {'min':0,   'max':0.50, 'palette':['#ffffcc','#fd8d3c','#bd0026']}
UQ_VIS_BOOTSTRAP = {'min':0,   'max':0.30, 'palette':['#ffffcc','#fd8d3c','#bd0026']}
CONF_VIS         = {'min':0,   'max':3,    'palette': CONF_PALETTE}
CONF_ZONE_VIS    = {'min':0,   'max':1,    'palette':['#f0f0f0','#d01c8b']}


def add_all_layers(Map: geemap.Map,
                   roi: ee.Geometry,
                   district_fc: ee.FeatureCollection,
                   district_name: str,
                   rf_prob: ee.Image,
                   rf_class: ee.Image,
                   uq_results: dict,
                   all_predictors: ee.Image,
                   sel_vars: ee.List,
                   flood_pts: ee.FeatureCollection,
                   non_flood_pts: ee.FeatureCollection):
    """
    Add all layers to the geemap Map.
    Primary outputs (FSM prob, 5-class, UQ layers) are shown by default.
    Conditioning factor layers are hidden — toggle via the Layers panel.
    """
    prob_vis  = {'min':0,'max':1,'palette':FSM_PALETTE}
    class_vis = {'min':1,'max':5,'palette':FSM_PALETTE}

    boundary = ee.Image().byte().paint(district_fc,1,3)
    Map.addLayer(boundary, {'palette':'#000000'}, f'{district_name} Boundary', True, 0.9)

    # ── Primary FSM layers ──────────────────────────────────────────────
    Map.addLayer(rf_prob,  prob_vis,  '🔴 RF Susceptibility Probability', True)
    Map.addLayer(rf_class, class_vis, '🟠 RF FSM (5-class)',              False)

    # ── UQ layers ───────────────────────────────────────────────────────
    Map.addLayer(uq_results['uq_variance'],           UQ_VIS_VARIANCE,  '📊 UQ-1: Bernoulli Variance',   False)
    Map.addLayer(uq_results['uq_margin'],             UQ_VIS_MARGIN,    '📊 UQ-2: Margin Confidence',    False)
    Map.addLayer(uq_results['uq_bootstrap'],          UQ_VIS_BOOTSTRAP, '📊 UQ-3: Bootstrap Std Dev',    False)
    Map.addLayer(uq_results['conformal_pred_set'],    CONF_VIS,         '📊 UQ-4: Conformal Pred Set',   False)
    Map.addLayer(uq_results['conformal_uncertain_zone'],CONF_ZONE_VIS,  '📊 UQ-4: Conformal Uncertain Zone', False)

    # ── Flood inventory ─────────────────────────────────────────────────
    Map.addLayer(flood_pts,     {'color':'red'},  '🔴 Flood Points',     False)
    Map.addLayer(non_flood_pts, {'color':'blue'}, '🔵 Non-Flood Points', False)

    # ── Selected conditioning factors (hidden) ───────────────────────────
    for v in sel_vars.getInfo():
        if v in LAYER_VIS:
            Map.addLayer(all_predictors.select([v]).toFloat(),
                          LAYER_VIS[v], f'📌 {v}', False)

    Map.centerObject(roi, 10)
    print(f"  [add_all_layers] Layers added. Toggle visibility in the Layers panel.")

print("✅ LAYER_VIS and add_all_layers() defined.")


## 🖥️ Cell 15 — Interactive Widget UI

Run this cell to display the control panel.
Select **State → District**, adjust sliders, then click **▶ Run Analysis**.


In [ ]:
S = {'description_width':'170px'}
W = widgets.Layout(width='100%')

# ── Study area ───────────────────────────────────────────────────────────
state_w = widgets.Dropdown(
    options=list(DISTRICT_CONFIG.keys()), value='Karnataka',
    description='🌏 State:', style=S, layout=W)
district_w = widgets.Dropdown(
    options=list(DISTRICT_CONFIG['Karnataka'].keys()), value='Dakshina Kannada',
    description='📍 District:', style=S, layout=W)

def _on_state(change):
    if change['name'] == 'value':
        opts = list(DISTRICT_CONFIG[change['new']].keys())
        district_w.options = opts
        district_w.value   = opts[0]
state_w.observe(_on_state, names='value')

# ── Variable selection ───────────────────────────────────────────────────
top_n_w = widgets.IntSlider(
    value=10, min=5, max=17, step=1,
    description='🔢 Top-N Variables:', style=S, layout=W)

widgets.HTML(
    '<i style="color:#555;font-size:11px">'
    'SAR is excluded from predictors (used only for flood inventory sampling).'
    '</i>')

# ── Sample sizes ─────────────────────────────────────────────────────────
n_flood_w    = widgets.IntSlider(value=500, min=100,max=2000,step=100,
                                  description='# Flood Points:',   style=S, layout=W)
n_nonflood_w = widgets.IntSlider(value=500, min=100,max=2000,step=100,
                                  description='# Non-Flood Pts:',  style=S, layout=W)

# ── RF hyperparameters ───────────────────────────────────────────────────
preset_w = widgets.ToggleButtons(
    options=['Default','Tuned','Custom'], value='Tuned',
    description='⚙️ Preset:',
    style={'description_width':'80px','button_width':'72px'})

rf_trees_w    = widgets.IntSlider(  value=500,  min=100,max=1000,step=50,  description='RF Trees:',       style=S,layout=W)
rf_varsplit_w = widgets.IntSlider(  value=4,    min=2,  max=17,  step=1,   description='RF Vars/Split:',  style=S,layout=W)
rf_minleaf_w  = widgets.IntSlider(  value=5,    min=1,  max=20,  step=1,   description='RF Min Leaf:',    style=S,layout=W)
rf_bag_w      = widgets.FloatSlider(value=0.6,  min=0.4,max=0.9, step=0.05,description='RF Bag Fraction:',style=S,layout=W)
custom_box = widgets.VBox([
    widgets.HTML('<b style="color:#1565c0">── Custom RF Parameters ──</b>'),
    rf_trees_w, rf_varsplit_w, rf_minleaf_w, rf_bag_w,
], layout=widgets.Layout(border='1px solid #bbb',padding='8px',margin='4px 0',border_radius='4px'))
custom_box.layout.display = 'none'

def _toggle(change):
    custom_box.layout.display = '' if change['new']=='Custom' else 'none'
preset_w.observe(_toggle, names='value')

# ── UQ parameters ────────────────────────────────────────────────────────
n_bootstrap_w = widgets.IntSlider(
    value=5, min=3, max=15, step=1,
    description='🔁 Bootstrap K:',
    style=S, layout=W)
alpha_w = widgets.FloatSlider(
    value=0.10, min=0.05, max=0.30, step=0.01,
    description='α (conformal):',
    style=S, layout=W,
    readout_format='.2f')

# ── Run button ───────────────────────────────────────────────────────────
run_btn    = widgets.Button(
    description='▶  Run FloodSight-UQ Analysis',
    button_style='success',
    layout=widgets.Layout(width='100%',height='42px',margin='10px 0 4px 0'))
run_btn.style.font_weight = 'bold'
status_lbl = widgets.HTML(
    '<span style="color:#388e3c;font-size:12px">✅ Ready.</span>')
out_area   = widgets.Output()

_widget_refs = {
    'rf_trees':rf_trees_w,'rf_varsplit':rf_varsplit_w,
    'rf_minleaf':rf_minleaf_w,'rf_bag':rf_bag_w,
}

ctrl_panel = widgets.VBox([
    widgets.HTML(
        '<div style="background:#1565c0;padding:12px;border-radius:6px;margin-bottom:8px">'
        '<span style="color:white;font-size:17px;font-weight:bold">🌊 FloodSight-UQ</span><br>'
        '<span style="color:#bbdefb;font-size:11px">'
        'West Coast India · RF · 4-Method UQ · Conformal Prediction</span></div>'),
    widgets.HTML('<b>📍 Study Area</b>'),
    state_w, district_w,
    widgets.HTML('<b>🔍 Variables &amp; Samples</b>'),
    top_n_w,
    widgets.HTML(
        '<span style="color:#777;font-size:11px;font-style:italic">'
        '⚠ SAR excluded from predictors (used for flood inventory only)</span>'),
    n_flood_w, n_nonflood_w,
    widgets.HTML('<b>🤖 Random Forest</b>'),
    preset_w, custom_box,
    widgets.HTML('<b>🔬 Uncertainty Quantification</b>'),
    n_bootstrap_w, alpha_w,
    run_btn, status_lbl,
], layout=widgets.Layout(width='500px',border='2px solid #cfd8dc',
                          padding='12px',border_radius='8px'))

display(ctrl_panel)
display(out_area)
print("✅ Widget UI displayed — select State/District and click Run.")


## 🚀 Cell 16 — Main Analysis Runner

Run this cell once to register the button callback, then click **▶ Run** in the panel above.
All seven pipeline steps are logged to the output area in real time.

```
Step 1 → Load district boundary (FAO GAUL)
Step 2 → Build 17-band conditioning factor image
Step 3 → Build flood inventory (GFD + SAR, labels only)
Step 4 → Rank variables by RF importance → select top-N  [3-way split]
Step 5 → Train final RF model
Step 6 → Compute FSM probability map + 5-class reclassification
Step 7 → Compute 4 UQ layers (Bernoulli · Margin · Bootstrap · Conformal)
       → Accuracy assessment
       → Plots (importance, UQ histograms, conformal calibration, area stats)
       → Interactive geemap
```


In [ ]:
_RESULTS = {}   # populated after each run; shared with export cell

def run_analysis(_evt=None):
    with out_area:
        clear_output(wait=True)

        # ── Read widgets ──────────────────────────────────────────────────
        state_name    = state_w.value
        district_name = district_w.value
        top_n         = int(top_n_w.value)
        n_flood       = int(n_flood_w.value)
        n_non_flood   = int(n_nonflood_w.value)
        preset        = preset_w.value
        n_bootstrap   = int(n_bootstrap_w.value)
        alpha         = float(alpha_w.value)
        rf_params     = get_rf_params(preset, _widget_refs)

        cfg       = DISTRICT_CONFIG[state_name][district_name]
        gaul_name = cfg['gaul']

        status_lbl.value = (f'<span style="color:#e65100;font-size:12px">'
                             f'⏳ Running — {district_name}, {state_name}...</span>')
        print(f"{'═'*64}")
        print(f"  🌊 FloodSight-UQ  |  {district_name}, {state_name}")
        print(f"  Preset: {preset} | Top-N: {top_n} | K_boot: {n_bootstrap} | α={alpha}")
        print(f"{'═'*64}")

        # ── Step 1: ROI ───────────────────────────────────────────────────
        print("\n[1/7] Loading district boundary from FAO GAUL 2015...")
        district_fc = (ee.FeatureCollection('FAO/GAUL/2015/level2')
                       .filter(ee.Filter.and_(
                           ee.Filter.eq('ADM2_NAME', gaul_name),
                           ee.Filter.eq('ADM1_NAME', state_name))))
        if district_fc.size().getInfo() == 0:
            print(f"  ⚠️  State-filtered query empty — trying ADM2_NAME only...")
            district_fc = (ee.FeatureCollection('FAO/GAUL/2015/level2')
                           .filter(ee.Filter.eq('ADM2_NAME', gaul_name)))
        if district_fc.size().getInfo() == 0:
            status_lbl.value = (f'<span style="color:red">❌ GAUL name "{gaul_name}" '
                                  f'not found. Run Cell 5.</span>')
            print(f"  ❌ GAUL name '{gaul_name}' not found — run Cell 5 to verify.")
            return
        roi = district_fc.first().geometry()
        print(f"  ✅ ROI loaded (GAUL: '{gaul_name}')")

        # ── Step 2: Conditioning factors (17 bands, SAR-free) ────────────
        print("\n[2/7] Building 17-band conditioning factor image (SAR excluded)...")
        all_predictors = build_conditioning_factors(roi)

        # ── Step 3: Flood inventory ───────────────────────────────────────
        print("\n[3/7] Building flood inventory (GFD + SAR labels)...")
        flood_pts, non_flood_pts, _ = build_flood_inventory(
            roi, n_flood=n_flood, n_non_flood=n_non_flood)

        # ── Step 4: Variable selection + 3-way split ──────────────────────
        print(f"\n[4/7] Ranking variables → selecting top-{top_n}...")
        (sel_preds, train_sel, cal_sel, test_sel,
         sel_vars, imp_dict) = prepare_samples(
             all_predictors, flood_pts, non_flood_pts, top_n)

        # ── Step 5: Train RF ──────────────────────────────────────────────
        print("\n[5/7] Training RF classifier...")
        rf_clfs = train_rf(train_sel, sel_vars, rf_params)

        # ── Step 6: Susceptibility map ────────────────────────────────────
        print("\n[6/7] Computing RF susceptibility map + FSM reclassification...")
        rf_prob, rf_class = compute_fsm(rf_clfs, sel_preds)

        # ── Step 7: UQ + accuracy + plots ────────────────────────────────
        print("\n[7/7] Computing 4-method uncertainty quantification...")
        uq = compute_uq_all(rf_clfs, sel_preds, train_sel, cal_sel, test_sel,
                              sel_vars, rf_params, n_bootstrap=n_bootstrap, alpha=alpha)

        print("\n  Accuracy assessment...")
        acc_df = assess_accuracy_rf(rf_clfs, test_sel)
        print_accuracy_table(acc_df, f"{district_name}, {state_name}")

        # ── Store for export cell ─────────────────────────────────────────
        _RESULTS.update(dict(
            state=state_name, district=district_name,
            roi=roi, district_fc=district_fc,
            all_predictors=all_predictors,
            sel_preds=sel_preds, sel_vars=sel_vars,
            rf_prob=rf_prob, rf_class=rf_class,
            uq=uq, acc_df=acc_df,
            flood_pts=flood_pts, non_flood_pts=non_flood_pts,
            imp_dict=imp_dict, rf_clfs=rf_clfs,
            train_sel=train_sel, cal_sel=cal_sel, test_sel=test_sel,
        ))

        # ── Plots ─────────────────────────────────────────────────────────
        print("\n📊 Variable importance chart...")
        plot_variable_importance(imp_dict, sel_vars, f"{district_name}, {state_name}")

        print("\n📈 UQ histograms + summary statistics...")
        plot_uq_histograms(uq, roi, rf_prob, f"{district_name}, {state_name}")

        print("\n📉 Conformal calibration curve...")
        plot_conformal_calibration(uq['cal_nc_scores'], uq['test_nc_scores'],
                                    f"{district_name}, {state_name}")

        print("\n🗂️ FSM area distribution...")
        plot_area_distribution(rf_class, roi, f"{district_name}, {state_name}")

        # ── Interactive map ───────────────────────────────────────────────
        print("\n🗺️ Building interactive geemap...")
        Map = geemap.Map(center=cfg['center'], zoom=cfg['zoom'])
        Map.add_basemap('HYBRID')
        add_all_layers(Map, roi, district_fc, district_name,
                        rf_prob, rf_class, uq,
                        all_predictors, sel_vars, flood_pts, non_flood_pts)

        Map.add_colorbar(
            vis_params={'min':1,'max':5,'palette':FSM_PALETTE},
            label='Flood Susceptibility Class',
            layer_name='RF FSM (5-class)')

        # Add conformal legend
        conf_labels = ['Empty (anomaly)','Non-flood only','Flood only','Both (uncertain)']
        patches = [mpatches.Patch(color=CONF_PALETTE[i], label=conf_labels[i])
                   for i in range(4)]
        _RESULTS['Map'] = Map
        display(Map)

        status_lbl.value = (f'<span style="color:#2e7d32;font-size:12px">'
                             f'✅ Done — {district_name}, {state_name}. '
                             f'Coverage={uq["empirical_coverage"]:.4f}  '
                             f'(target ≥ {1-alpha:.2f})  '
                             f'Run Cell 17 to export.</span>')
        print(f"\n✅ Analysis complete — {district_name}, {state_name}!")


run_btn.on_click(run_analysis)
print("✅ Callback registered. Click ▶ Run FloodSight-UQ Analysis in the widget above.")


## 💾 Cell 17 — Export to Google Drive

| File | Format | Content |
|------|--------|---------|
| `RF_Susceptibility.tif` | GeoTIFF 100 m | RF probability [0,1] |
| `RF_FSM_5class.tif` | GeoTIFF 100 m | 5-class susceptibility map |
| `UQ1_Bernoulli_Variance.tif` | GeoTIFF 100 m | p(1−p) |
| `UQ2_Margin_Confidence.tif` | GeoTIFF 100 m | min(p,1−p) |
| `UQ3_Bootstrap_StdDev.tif` | GeoTIFF 100 m | Bootstrap StdDev |
| `UQ4_Conformal_PredSet.tif` | GeoTIFF 100 m | Categorical 0-3 |
| `UQ4_Conformal_UncertainZone.tif` | GeoTIFF 100 m | Binary uncertain zone |
| `CF_<band>.tif` (×top-N) | GeoTIFF 100 m | Individual conditioning factors |
| `conformal_scores.csv` | CSV | Cal + test nonconformity scores |
| `accuracy_metrics.csv` | CSV | RF performance metrics |

> After clicking Export, visit the **Tasks** tab at [code.earthengine.google.com/tasks](https://code.earthengine.google.com/tasks) and click **Run** for each pending task.


In [ ]:
def _safe_name(s: str) -> str:
    return s.replace(' ','_').replace('(','').replace(')','')

def export_tif(image: ee.Image, name: str, roi: ee.Geometry,
               folder: str, scale: int = 100):
    task = ee.batch.Export.image.toDrive(
        image          = image.toFloat(),
        description    = name[:100],
        folder         = folder,
        fileNamePrefix = name,
        region         = roi, scale=scale,
        crs            = 'EPSG:4326',
        maxPixels      = 1_000_000_000,
        fileFormat     = 'GeoTIFF',
    )
    task.start()
    return task


def run_exports():
    if not _RESULTS:
        print("⚠️  No results found — run Cell 16 first.")
        return

    state    = _RESULTS['state']
    district = _RESULTS['district']
    roi      = _RESULTS['roi']
    uq       = _RESULTS['uq']
    prefix   = f"{state}_{_safe_name(district)}"
    folder   = f"FSM_WestCoast/{state}/{_safe_name(district)}"

    print(f"{'═'*60}")
    print(f"  💾 Exporting — {district}, {state}")
    print(f"  Drive folder: {folder}")
    print(f"{'═'*60}")

    tasks = {}

    # ── FSM maps ──────────────────────────────────────────────────────────
    tasks['RF_Prob']    = export_tif(_RESULTS['rf_prob'],  f"{prefix}_RF_Susceptibility",   roi, folder)
    tasks['RF_FSM']     = export_tif(_RESULTS['rf_class'], f"{prefix}_RF_FSM_5class",        roi, folder)

    # ── UQ layers ─────────────────────────────────────────────────────────
    tasks['UQ1'] = export_tif(uq['uq_variance'],           f"{prefix}_UQ1_Bernoulli_Variance", roi, folder)
    tasks['UQ2'] = export_tif(uq['uq_margin'],             f"{prefix}_UQ2_Margin_Confidence",  roi, folder)
    tasks['UQ3'] = export_tif(uq['uq_bootstrap'],          f"{prefix}_UQ3_Bootstrap_StdDev",   roi, folder)
    tasks['UQ4_set']  = export_tif(uq['conformal_pred_set'],
                                    f"{prefix}_UQ4_Conformal_PredSet",    roi, folder)
    tasks['UQ4_zone'] = export_tif(uq['conformal_uncertain_zone'],
                                    f"{prefix}_UQ4_Conformal_UncertainZone", roi, folder)

    for k, t in tasks.items():
        print(f"  📤 Submitted: {k}")

    # ── Conditioning factors (individual bands) ───────────────────────────
    var_list = _RESULTS['sel_vars'].getInfo()
    for v in var_list:
        single = _RESULTS['sel_preds'].select([v]).toFloat()
        t = export_tif(single, f"{prefix}_CF_{v}", roi, folder)
        print(f"  📤 CF: {v}")

    # ── CSV: conformal nonconformity scores ───────────────────────────────
    import csv, io
    cal_scores  = uq['cal_nc_scores']
    test_scores = uq['test_nc_scores']
    max_len     = max(len(cal_scores), len(test_scores))
    csv_path    = f'/content/{prefix}_conformal_scores.csv'
    with open(csv_path,'w',newline='') as f:
        w = csv.writer(f)
        w.writerow(['cal_nc_score','test_nc_score'])
        for i in range(max_len):
            w.writerow([
                cal_scores[i]  if i < len(cal_scores)  else '',
                test_scores[i] if i < len(test_scores) else '',
            ])
    print(f"\n  📊 Conformal scores saved: {csv_path}")

    # ── CSV: accuracy metrics ─────────────────────────────────────────────
    acc_path = f'/content/{prefix}_accuracy_metrics.csv'
    _RESULTS['acc_df'].to_csv(acc_path)
    print(f"  📊 Accuracy metrics saved: {acc_path}")

    print(f"\n✅ {len(tasks)+len(var_list)} GEE export tasks submitted.")
    print(f"   Visit https://code.earthengine.google.com/tasks to confirm.")
    return tasks


export_btn = widgets.Button(
    description='💾  Export All to Drive',
    button_style='info',
    layout=widgets.Layout(width='380px',height='40px'))
export_btn.style.font_weight = 'bold'
export_out = widgets.Output()

def _on_export(_):
    with export_out:
        clear_output(wait=True)
        run_exports()

export_btn.on_click(_on_export)
display(widgets.VBox([
    widgets.HTML('<b>Export results for the last-run district:</b>'),
    export_btn, export_out]))


## 📉 Cell 18 — ROC Curve & AUC *(Post-Analysis)*

Fetches test-set predicted probabilities directly from GEE and plots the RF ROC curve.
No Drive download needed.  Re-run after Cell 16 completes.


In [ ]:
def plot_roc_curve(rf_clfs: dict,
                   test_sel: ee.FeatureCollection,
                   band_names: ee.List,
                   district_name: str):
    """Sample RF test-set probabilities and plot ROC with AUC."""
    print("  [plot_roc_curve] Fetching test predictions (~30 s)...")
    rows = (test_sel.classify(rf_clfs['prob'])
            .select(['classification','label'])
            .getInfo()['features'])
    if not rows:
        print("  ⚠️  No test predictions — rerun analysis first.")
        return

    probs  = np.array([r['properties']['classification'] for r in rows])
    labels = np.array([r['properties']['label']          for r in rows], dtype=int)

    thresholds = np.linspace(0,1,300)
    tprs,fprs  = [],[]
    for thr in thresholds:
        pred = (probs >= thr).astype(int)
        tp = np.sum((pred==1)&(labels==1)); fn = np.sum((pred==0)&(labels==1))
        fp = np.sum((pred==1)&(labels==0)); tn = np.sum((pred==0)&(labels==0))
        tprs.append(tp/(tp+fn+1e-9))
        fprs.append(fp/(fp+tn+1e-9))

    pairs  = sorted(zip(fprs,tprs))
    fpr_s  = [p[0] for p in pairs]
    tpr_s  = [p[1] for p in pairs]
    auc    = np.trapz(tpr_s, fpr_s)

    # Also compute AUC at conformal q threshold
    q = _RESULTS.get('uq',{}).get('q_threshold', None)

    fig, ax = plt.subplots(figsize=(7,6))
    ax.plot([0,1],[0,1],'k--',lw=1.2,label='Random (AUC=0.50)')
    ax.plot(fpr_s, tpr_s, color='#1565c0', lw=2.5, label=f'RF  (AUC={auc:.4f})')
    if q is not None:
        # Mark the operating point corresponding to the conformal q threshold
        op_idx = np.argmin(np.abs(thresholds-(1-q)))
        ax.scatter([fprs[op_idx]],[tprs[op_idx]],s=80,zorder=5,
                    color='#d32f2f',label=f'Conformal op. pt (q={q:.3f})')

    ax.set_xlabel('False Positive Rate (1−Specificity)',fontsize=12)
    ax.set_ylabel('True Positive Rate (Sensitivity)',   fontsize=12)
    ax.set_title(f'ROC Curve — {district_name}',fontweight='bold',fontsize=13)
    ax.legend(loc='lower right',fontsize=10)
    ax.set_xlim([-0.02,1.02]); ax.set_ylim([-0.02,1.05])
    ax.grid(True,alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  RF AUC = {auc:.4f}")


if _RESULTS:
    plot_roc_curve(_RESULTS['rf_clfs'], _RESULTS['test_sel'],
                   _RESULTS['sel_vars'],
                   f"{_RESULTS['district']}, {_RESULTS['state']}")
else:
    print("⚠️  Run Cell 16 first, then re-run this cell.")


## 🔄 Cell 19 — Batch Runner *(Optional)*

Programmatic loop over any list of (state, district) pairs — no widget interaction needed.
Designed for overnight or large-scale production runs.
All accuracy results are collected into a single summary CSV.


In [ ]:
BATCH_LIST = [
    ('Karnataka',  'Dakshina Kannada'),
    ('Karnataka',  'Udupi'),
    ('Karnataka',  'Uttara Kannada'),
    ('Kerala',     'Kasaragod'),
    ('Kerala',     'Kannur'),
    # Add more tuples as needed
]

BATCH_PARAMS = dict(
    top_n=10, n_flood=500, n_non_flood=500,
    preset='Tuned', n_bootstrap=5, alpha=0.10,
)


def run_batch(batch_list: list, params: dict = BATCH_PARAMS):
    """
    Run the full RF + 4-method UQ pipeline for every (state, district) pair.
    Exports Ensemble Susceptibility and all UQ TIFFs automatically.
    Returns a summary DataFrame of accuracy + conformal coverage per district.
    """
    hp      = get_rf_params(params['preset'],{})
    summary = []

    for idx,(state_name,district_name) in enumerate(batch_list):
        print(f"\n{'═'*64}")
        print(f"  [{idx+1}/{len(batch_list)}]  {district_name}, {state_name}")
        print(f"{'═'*64}")

        if (state_name not in DISTRICT_CONFIG or
                district_name not in DISTRICT_CONFIG[state_name]):
            print(f"  ⚠️  Not in DISTRICT_CONFIG — skipping."); continue

        cfg       = DISTRICT_CONFIG[state_name][district_name]
        gaul_name = cfg['gaul']

        try:
            district_fc = (ee.FeatureCollection('FAO/GAUL/2015/level2')
                           .filter(ee.Filter.and_(
                               ee.Filter.eq('ADM2_NAME',gaul_name),
                               ee.Filter.eq('ADM1_NAME',state_name))))
            if district_fc.size().getInfo()==0:
                district_fc = (ee.FeatureCollection('FAO/GAUL/2015/level2')
                               .filter(ee.Filter.eq('ADM2_NAME',gaul_name)))
            if district_fc.size().getInfo()==0:
                print(f"  ❌ GAUL '{gaul_name}' not found — skipping."); continue
            roi = district_fc.first().geometry()

            all_preds = build_conditioning_factors(roi)
            fp,nfp,_  = build_flood_inventory(roi, params['n_flood'], params['n_non_flood'])
            (sel_preds,train_sel,cal_sel,test_sel,
             sel_vars,imp_d) = prepare_samples(all_preds,fp,nfp,params['top_n'])
            rf_clfs   = train_rf(train_sel,sel_vars,hp)
            rf_prob,rf_class = compute_fsm(rf_clfs,sel_preds)
            uq        = compute_uq_all(rf_clfs,sel_preds,train_sel,cal_sel,test_sel,
                                        sel_vars,hp,params['n_bootstrap'],params['alpha'])
            acc_df    = assess_accuracy_rf(rf_clfs,test_sel)
            print_accuracy_table(acc_df,f"{district_name}, {state_name}")

            row = {'State':state_name,'District':district_name,
                   **acc_df.iloc[0].to_dict(),
                   'Conformal_Coverage':uq['empirical_coverage'],
                   'q_threshold':uq['q_threshold']}
            summary.append(row)

            # Submit key exports
            folder = f"FSM_WestCoast/{state_name}/{_safe_name(district_name)}"
            prefix = f"{state_name}_{_safe_name(district_name)}"
            for img,nm in [(rf_prob,'RF_Susceptibility'),
                            (rf_class,'RF_FSM_5class'),
                            (uq['uq_variance'],'UQ1_Bernoulli_Variance'),
                            (uq['uq_margin'],'UQ2_Margin_Confidence'),
                            (uq['uq_bootstrap'],'UQ3_Bootstrap_StdDev'),
                            (uq['conformal_pred_set'],'UQ4_Conformal_PredSet'),
                            (uq['conformal_uncertain_zone'],'UQ4_Conformal_UncertainZone')]:
                export_tif(img, f"{prefix}_{nm}", roi, folder)
            print(f"  📤 7 export tasks submitted for {district_name}.")

        except Exception as exc:
            print(f"  ❌ Error for {district_name}: {exc}")

    if summary:
        df = pd.DataFrame(summary)
        csv_path = '/content/batch_summary.csv'
        df.to_csv(csv_path, index=False)
        print(f"\n✅ Batch complete. Summary → {csv_path}")
        display(df.set_index(['State','District']))
    return summary


# ─── Uncomment to run ───────────────────────────────────────────────
# run_batch(BATCH_LIST)
print("💡 Edit BATCH_LIST, then uncomment run_batch(BATCH_LIST) to start.")
